In [ ]:
from catboost import CatBoostClassifier
import catboost as catboost
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from random import sample
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, mean_absolute_error, r2_score, roc_auc_score, roc_curve, cohen_kappa_score, brier_score_loss
from sklearn.metrics import matthews_corrcoef
#import shap

In [ ]:
data = pd.read_excel("/home/acdsd/Documents/TNBC/Atompair_FP_Trainset_DD.xlsx")
data

In [ ]:
df = pd.DataFrame(data)
df.reset_index(drop=True, inplace=True)
#df

In [ ]:
df.Target.value_counts()

In [ ]:
y_train = df.Target
x_train = df.drop('Target', axis=1)

Split the dataset

In [ ]:
y_train.value_counts()

In [ ]:
#Read the validation set
v_test_data = pd.read_excel("/home/acdsd/Documents/TNBC/Atompair_FP_Testset_DD.xlsx")
v_test_data = pd.DataFrame(v_test_data)
v_test_data.shape

In [ ]:
Y_v_test_data = v_test_data.Target
X_v_test_data = v_test_data.drop('Target', axis=1)

In [ ]:
cb = catboost.CatBoostClassifier(
    learning_rate = 0.015424968015265779,
    iterations=501,
    random_seed= 88,
    depth = 8,
    l2_leaf_reg= 5.102211173868346
)
cb.fit(x_train, y_train, verbose=False, plot=True)

In [ ]:
v_predictions = cb.predict(X_v_test_data)
acc_train=cb.score(x_train,y_train)*100
v_acc_test = cb.score(X_v_test_data, Y_v_test_data)*100
print("Training accuracy with hyperparameters:",acc_train)
print("Validation accuracy with hyperparameters:",v_acc_test)

In [ ]:
from sklearn.metrics import matthews_corrcoef
print('Accuracy:%0.3f'% v_acc_test)
tn, fp, fn, tp = confusion_matrix(Y_v_test_data, v_predictions).ravel()
specificity = tn / (tn+fp)
sensitivity = tp / (tp+fn)
print('Sensitivity:%0.3f'% sensitivity)
print('Specificity:%0.3f'% specificity)
ba = 0.5 * (sensitivity + specificity)
print('Balance accuracy:%0.3f'% ba)
mcc = matthews_corrcoef(Y_v_test_data, v_predictions)
#mcc = matthews_corrcoef(Y_test_data, v_predictions)
print('MCC: %0.3f'% mcc)
r_auc_score = roc_auc_score(Y_v_test_data, v_predictions)
#r_auc_score = roc_auc_score(Y_test_data, v_predictions)
print('AUC: %0.3f' % r_auc_score)
bloss = brier_score_loss(Y_v_test_data, v_predictions)
#bloss = brier_score_loss(Y_test_data, v_predictions)
print('Brier_loss: %0.3f' % bloss)
FPR = fp/(fp+tn)
FNR = fn/(tp+fn)
Precision = tp/(tp+fp)
Recall = tp/(tp+fn)
print('False Postive rate: %0.3f' %FPR)
print('False Negative rate: %0.3f' %FNR)
print('Precision: %0.3f' %Precision)
print('Recall: %0.3f' %Recall)
f1=2*((Precision*Recall)/(Precision+Recall))
print('F1: %0.3f' %f1)
#kappa = cohen_kappa_score(Y_test_data, v_predictions)
kappa = cohen_kappa_score(Y_v_test_data, v_predictions)
print('Kappa: %0.3f' %kappa)

In [ ]:
cf =(confusion_matrix(Y_v_test_data, v_predictions))
print(classification_report(Y_v_test_data, v_predictions))

In [ ]:
cf_plt =sns.heatmap(cf,annot=True,cmap="Blues",fmt="d",cbar=False, annot_kws={"size": 24})
cf_plt.set(xlabel = "Predicted Value", ylabel ="True Value")
cf_plt

In [ ]:
fig = cf_plt.get_figure()
fig.savefig("/home/acdsd/Documents/TNBC/CatBoost_ATOMPAIR_FP/CB_Con_mat.png")

In [ ]:
r_probs = [0 for _ in range(len(Y_v_test_data))]
cb_prob = cb.predict_proba(X_v_test_data)
cb_prob = cb_prob[:,1]
#cb_prob

In [ ]:
r_auc_score = roc_auc_score(Y_v_test_data, cb_prob)
r_auc_score_1 = roc_auc_score(Y_v_test_data,r_probs)
print(r_auc_score)
fpr, tpr, _ = roc_curve(Y_v_test_data, cb_prob)
rfpr, rtpr, _ = roc_curve(Y_v_test_data, r_probs)

In [ ]:
plt.figure(figsize=(7, 5), dpi=600)
plt.plot(fpr, tpr, marker='.', label='CatBoost_classifier_1024FP (AUC = %0.3f)' % r_auc_score)
plt.plot(rfpr, rtpr, marker='_' % r_auc_score_1)
plt.xlabel('False Positive Rate -->')
plt.ylabel('True Positive Rate -->')
plt.legend()
plt.show()
plt.savefig('/home/acdsd/Documents/TNBC/CatBoost_ATOMPAIR_FP/CB_AUC_1024.png', dpi=600, bbox_inches='tight')

In [ ]:
plt.savefig('/home/acdsd/Documents/TNBC/CatBoost_ATOMPAIR_FP/CB_AUC_1024.png', dpi=600, bbox_inches='tight')

## Cross validation AUC

In [ ]:
from sklearn.metrics import RocCurveDisplay

In [ ]:
from scipy import interp
from sklearn.metrics import roc_curve,auc
cv = StratifiedKFold(n_splits=10,shuffle=False)

In [ ]:
# plot k fold ROC
plt.figure(figsize=(7, 5), dpi=600)
x_train = x_train.T
fig1 = plt.figure(figsize=[12,12])
ax1 = fig1.add_subplot(111,aspect = 'equal')
tprs = []
aucs = []
mean_fpr = np.linspace(0,1,100)
i = 1
for train,test in cv.split(x_train,y_train):
    prediction = cb.fit(x_train.iloc[train],y_train.iloc[train]).predict_proba(x_train.iloc[test])
    fpr, tpr, t = roc_curve(y_train.iloc[test], prediction[:, 1])
    tprs.append(interp(mean_fpr, fpr, tpr))
    roc_auc = auc(fpr, tpr)
    aucs.append(roc_auc)
    plt.plot(fpr, tpr, lw=2, alpha=0.3, label='ROC fold %d (AUC = %0.2f)' % (i, roc_auc))
    i= i+1

plt.plot([0,1],[0,1],linestyle = '--',lw = 2,color = 'black')
mean_tpr = np.mean(tprs, axis=0)
mean_auc = auc(mean_fpr, mean_tpr)
plt.plot(mean_fpr, mean_tpr, color='blue',
         label=r'Mean ROC (AUC = %0.2f )' % (mean_auc),lw=2, alpha=1)

plt.xlabel('False Positive Rate', fontsize = 20)
plt.ylabel('True Positive Rate', fontsize = 20)
plt.legend(loc="lower right")
plt.show()
plt.savefig('/home/acdsd/Documents/TNBC/CatBoost_ATOMPAIR_FP/CrossVal-AUC_CB_1024.png', dpi=600, bbox_inches='tight')

In [ ]:
import pickle

# Save the model to a .pkl file
with open('/home/acdsd/Documents/TNBC/Saved_model_TNBC/Catboost_Atompair_FP.pkl', 'wb') as file:
    pickle.dump(cb, file)

print("Model saved to Catboost_Atompair_FP.pkl")

In [ ]:
external_data = pd.read_excel("/home/acdsd/Documents/TNBC/IC50_compounds_1313/FingerPrint(FP)/Atom_Pair/Atom_Pair_fingerprints_1313.xlsx")
data_for_screening = pd.DataFrame(external_data)

In [ ]:
test_prob = cb.predict_proba(data_for_screening)
test_prob_F = pd.DataFrame(test_prob)
test_prob_F.to_csv('/home/acdsd/Documents/TNBC/New_1313_CatBoost_Atom_Pair_FP/Catboost_probability_AtompairFP.csv')

In [ ]:
accuracy_list = []
mcc_list = []
for i in range(50):
    y_train = y_train.sample(frac=1, replace=False, random_state=i)
    cb.fit(x_train, y_train)
    print('Trained')
    scrmb_predictions = cb.predict(X_v_test_data)
    print(i)
    accuracy = roc_auc_score(Y_v_test_data, scrmb_predictions)
    print('scra_pred')
    accuracy_list.append(accuracy)
    mcc_v = matthews_corrcoef(Y_v_test_data, scrmb_predictions)
    mcc_list.append(mcc_v)
print(accuracy_list)

In [ ]:
r_auc_score2 = roc_auc_score(Y_v_test_data, scrmb_predictions)
sns.set_style("white")
plt.figure(figsize = (20, 5), dpi=200)
ax = sns.distplot(accuracy_list, color="green")
plt.axvline(r_auc_score, color="green")
plt.xlabel("AUC Score", fontsize = 14)
plt.ylabel("Count", fontsize = 14)
ax.set(xlim=(0, 1))

In [ ]:
mcc2 = matthews_corrcoef(Y_v_test_data, scrmb_predictions)
sns.set_style("white")
plt.figure(figsize = (20, 5), dpi=200)
ax = sns.distplot(mcc_list, color="green")
plt.axvline(mcc, color="green")
plt.xlabel("MCC Score", fontsize = 14)
plt.ylabel("Count", fontsize = 14)
ax.set(xlim=(0, 1))

In [ ]:
def Average(lst):
    return sum(lst) / len(lst)
#print(mcc_list)
average = Average(accuracy_list)

# Printing average of the list
print("Average of the list =", round(average, 3))

In [ ]:
def Average(lst):
    return sum(lst) / len(lst)
#print(mcc_list)
average = Average(mcc_list)

# Printing average of the list
print("Average of the mcc list =", round(average, 3))